|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 0:</h2>|<h1>From a Program to a Model<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: the incident file<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

You finished Part 0. You know that a model is a function, what the loop
does with it, what attention computes, and what a GPU is good at. Now use
that knowledge from the outside, with only the symptoms.

Each ticket below is a real class of failure. Each ticket gives you a
**symptom** and some **evidence**. Some of the evidence is noise, as in a
real incident. Write four lines for each ticket:

1. **Root cause.** One sentence.
2. **The number that proves it.** Not "it looks like". A computation.
3. **The fix.**
4. **The guard.** A test, an assert or an alert that catches it next time.

Four rules:

- The tickets are **not** in the order of the notebooks. First you must find
  which idea the ticket needs.
- At least one ticket is **not a bug**. "Nothing is broken" is a valid answer
  only if a number proves it.
- Write your answer **before** you open the solution.
- Every ticket has a scratch cell. Most tickets need one computation.

This notebook needs no GPU.

**The on-call colleague.** In Claude Code, type `/incident 0.1` (or any
other ticket number) to work a ticket as a conversation. The colleague has
access to the system. Ask for a log, a measurement or an experiment, and it
answers with what the system shows. When you write your four lines, it tells
you which lines are weak, and it asks a question about each one. It does not
tell you the cause until you ask for the solution.

### The reference sheet

| GPU | Memory | Bandwidth | bf16 compute |
|---|---|---|---|
| A100 SXM 80GB | 80 GB | 2,039 GB/s | 312 TFLOP/s |
| L40S | 48 GB | 864 GB/s | 362 TFLOP/s |
| a 24 GB card | 24 GB | | |

| Model | Layers | Attention heads | KV heads | head_dim | hidden | MLP | vocab | parameters |
|---|---|---|---|---|---|---|---|---|
| Qwen3-1.7B | 28 | 16 | 8 | 128 | 2048 | 6144 | 151,936 | 1.72 B |
| Llama-3-8B | 32 | 32 | 8 | 128 | 4096 | 14336 | 128,256 | 8.03 B |

| Number format | Bytes | Largest value |
|---|---|---|
| float32 | 4 | 3.4 x 10^38 |
| bfloat16 | 2 | 3.4 x 10^38 |
| float16 | 2 | 65,504 |

In the Qwen3 vocabulary, token 0 is `!`.

Two facts from Part 0:

    ridge point (FLOP/byte) = compute / bandwidth
    a kernel is fast when it reaches the roof that limits it: bytes/s OR FLOP/s

# Ticket 1: the model speaks in tongues

**Severity:** high. **Reported by:** the QA team.

> Since the upgrade, every answer is a mix of random words in five
> languages. Sometimes the service crashes.

**Evidence**

- The team moved from an old Llama-2 model to Qwen3-1.7B. The config file
  has two keys, `model_path` and `tokenizer_path`. The upgrade changed
  `model_path`.
- A sample answer:

      Paris Unterstützung obtener 那么 tijdens ._ Vorlage

- The crash, one time in about twenty requests:

      IndexError: piece id is out of range.

- The logs also show a warning at every start:

      Setting `pad_token_id` to `eos_token_id` for open-end generation.

- The largest token id in the logged outputs of one hour is 151,398.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 2: the answer is only exclamation marks

**Severity:** medium. **Reported by:** a research engineer.

> I wrote my own attention to learn how it works, and I run the model in
> float16 to save memory. Short prompts work. Long prompts return
> `!!!!!!!!!!!!!!!!`.

**Evidence**

- The attention code:

  ```python
  scores = q @ k.transpose(-1, -2) / math.sqrt(head_dim)
  weights = torch.exp(scores)
  weights = weights / weights.sum(-1, keepdim=True)
  out = weights @ v
  ```

- The model is a small model that the team trained with the Qwen3
  tokenizer. The HF attention in float16 works on the same long prompts.
- A debug print of the largest score in the first layer: 9.8 for a short
  prompt, 14.2 for a long prompt.
- The failing prompts contain source code. The engineer thinks that the
  tokenizer handles code badly.
- For the long prompts, every value of the logits is `nan`.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 3: the test passes and the model fails

**Severity:** medium. **Reported by:** a developer.

> My attention function passes its unit test with a difference of 0.0.
> When I put it in the model, the model writes nonsense from the first
> token.

**Evidence**

- The function:

  ```python
  def my_attention(q, k, v):
      scores = q @ k.transpose(-1, -2) / math.sqrt(q.shape[-1])
      return torch.softmax(scores, dim=-1) @ v
  ```

- The test:

  ```python
  q = torch.randn(1, 16, 1, 128)     # one query
  k = torch.randn(1, 16, 50, 128)
  v = torch.randn(1, 16, 50, 128)
  want = F.scaled_dot_product_attention(q, k, v, is_causal=True)
  assert (my_attention(q, k, v) - want).abs().max() == 0
  ```

- The model runs the naive loop of Part 0: each step is a forward pass
  over the whole prefix.
- The developer thinks that the problem is the random tensors in the
  test, because they are not real activations.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 4: the kernel at 0.2% of peak

**Severity:** low. But it asks for two weeks of work. **Reported by:**
the performance team.

> Our RMSNorm kernel reaches 0.68 TFLOP/s on an L40S. The card can do
> 362. That is 0.19% of the peak. We want two engineers for two weeks to
> rewrite it.

**Evidence**

- The input is 8,192 tokens x 2,048 values in bfloat16. The output has
  the same shape. The weight is 2,048 values.
- One call takes 0.098 ms.
- RMSNorm does about 4 FLOP for each value: a square, an add, and two
  multiplies.
- The profiler says that the kernel uses no tensor cores.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 5: a 16 GB model does not fit in 24 GB

**Severity:** medium. **Reported by:** a new team.

> The model card says that Llama-3-8B needs 16 GB. Our card has 24 GB.
> The load fails with out of memory. Is the card broken?

**Evidence**

- The load code:

  ```python
  model = AutoModelForCausalLM.from_pretrained('meta-llama/Meta-Llama-3-8B').cuda()
  ```

- The container image has transformers 4.44. In that version,
  `from_pretrained` loads float32 when you do not give a dtype.
- The error:

      torch.OutOfMemoryError: CUDA out of memory. Tried to allocate
      224.00 MiB. GPU 0 has a total capacity of 23.68 GiB of which
      180.00 MiB is free. 22.10 GiB is allocated by PyTorch.

- A desktop session on the same card uses 400 MB.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 6: greedy is not greedy

**Severity:** medium. **Reported by:** the evaluation team.

> Our benchmark uses greedy decoding, but the score changes each time we
> run it: 61.2, 58.9, 60.4. We set
> `torch.use_deterministic_algorithms(True)`, and it did not help. The
> GPU is not deterministic.

**Evidence**

- The model is Qwen3-1.7B. The harness:

  ```python
  out = model.generate(input_ids, max_new_tokens=100)
  ```

- The `generation_config.json` of the model:

  ```json
  {"do_sample": true, "temperature": 0.6, "top_k": 20, "top_p": 0.95,
   "eos_token_id": [151645, 151643], "pad_token_id": 151643}
  ```

- Two runs of the same question often differ from the first or the
  second token.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 7: every answer is cut short

**Severity:** medium. **Reported by:** the users of a small internal
tool.

> The answers stop in the middle of a sentence. Long questions get
> almost no answer.

**Evidence**

- The model is an older model. Its `generation_config.json` sets no
  length. The service runs transformers 4.40. The code:

  ```python
  out = model.generate(input_ids)
  ```

- Some logged requests:

  | prompt tokens | new tokens |
  |---|---|
  | 9 | 11 |
  | 12 | 8 |
  | 15 | 5 |
  | 18 | 2 |

- The team thinks that the model stops early because it emits its
  end-of-sequence token too soon.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 8: the assistant finishes my question

**Severity:** high. **Reported by:** the product team.

> We ask "What is the capital of France?" and the assistant answers
> " What is the capital of Germany? What is the capital of Italy?". It
> does not answer. It continues the question.

**Evidence**

- The model is Qwen3-1.7B, the post-trained model, not `Qwen3-1.7B-Base`.
  The team checked.
- The code:

  ```python
  ids = tokenizer(question, return_tensors='pt').input_ids
  out = model.generate(ids, max_new_tokens=100, do_sample=False)
  ```

- The logged prompt ids for the question: 7 ids, from `3838` to `30`.
- The service uses a temperature of 0.7 in production. The team thinks
  that the temperature is too high.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 9: the vector add that fails for some sizes

**Severity:** low. **Reported by:** a student in the CUDA study group.

> My vector add kernel passes for 1,024, 4,096 and 65,536 elements. It
> fails for 1,000 and 5,000. I have the bounds check, so it cannot be an
> index problem.

**Evidence**

- The launch:

  ```cuda
  int threads = 256;
  int blocks = n / threads;
  add<<<blocks, threads>>>(a, b, out, n);
  ```

- The kernel:

  ```cuda
  __global__ void add(const float* a, const float* b, float* out, int n) {
      int i = blockIdx.x * blockDim.x + threadIdx.x;
      if (i < n) out[i] = a[i] + b[i];
  }
  ```

- For n = 1,000 the wrong elements are 768 to 999. They contain zeros.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

### Before you open the solution

Go back to each ticket and write one more line: **which piece of evidence was
noise, and why did it look relevant?**

Then go to Part 1. The tickets there use the arithmetic of the KV cache and
the roofline.